# 09 · Fehlerbehandlung

**Lernziele**

- Du liest einen Traceback und findest die Fehlerursache.
- Du fängst Fehler mit `try` / `except` / `else` / `finally` ab.
- Du löst selbst Fehler mit `raise` aus.
- Du kennst eigene Fehlerklassen und das `with`-Statement.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 2 und 9.

## Tracebacks lesen

Fehler sind normal. Wichtig ist, die Meldung zu lesen, und zwar **von unten nach oben**:

1. Letzte Zeile: Fehlertyp und Beschreibung
2. Darüber: die Zeile, in der es passiert ist
3. Noch weiter oben: der Weg dorthin (welche Funktion hat welche aufgerufen)

In [ ]:
def average(numbers):
    return sum(numbers) / len(numbers)


average([])

Häufige Fehlertypen:

| Fehler | Ursache | Beispiel |
| --- | --- | --- |
| `SyntaxError` | Code ist kein gültiges Python | fehlender Doppelpunkt |
| `NameError` | Name unbekannt | Tippfehler, Zelle nicht ausgeführt |
| `TypeError` | falscher Typ | `"a" + 1` |
| `ValueError` | richtiger Typ, falscher Wert | `int("drei")` |
| `IndexError` | Index außerhalb der Liste | `[1, 2][5]` |
| `KeyError` | Schlüssel fehlt im Dictionary | `{}["x"]` |
| `ZeroDivisionError` | Division durch 0 | `1 / 0` |
| `FileNotFoundError` | Datei existiert nicht | `open("gibtsnicht.txt")` |

## Fehler abfangen mit `try` / `except`

Code, der fehlschlagen kann, steht in `try`. Im passenden `except` reagierst du darauf.

In [ ]:
user_input = "drei"
try:
    number = int(user_input)
    print(f"Das Doppelte ist {number * 2}")
except ValueError:
    print(f"'{user_input}' ist keine Zahl")

Fange immer **konkrete** Fehlertypen ab. Ein nacktes `except:` verschluckt auch Fehler, die du sehen willst. *Errors should never pass silently.*

Mit `as` bekommst du das Fehlerobjekt:

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError as error:
    print(f"Rechenfehler: {error}")

**Übung 1 (Nacktes except verschluckt den Tippfehler, leicht):** Code, den eine KI geschrieben haben könnte (präpariert, ein Fehler eingebaut): Er meldet immer „Bitte eine ganze Zahl eingeben“, obwohl `quantity = "3"` gültig ist. Finde den Fehler und behebe ihn.

- Fange nur den Fehler ab, den du erwartest, und beobachte, was dann passiert.

In [ ]:
price = 4.20
quantity = "3"
try:
    total = int(quantity) * prise
    print(f"Summe: {total:.2f} €")
except:
    print("Bitte eine ganze Zahl eingeben")

In [ ]:
price = 4.20
quantity = "3"
try:
    total = int(quantity) * price  # vorher: prise -> NameError, vom nackten except verschluckt
except ValueError:
    print("Bitte eine ganze Zahl eingeben")
else:
    print(f"Summe: {total:.2f} €")
# Mit "except ValueError" und dem Tippfehler zeigt Python den NameError samt
# Vorschlag "Did you mean: 'price'?", statt ihn zu verschlucken.

## Mehrere Fehlertypen

Für verschiedene Fehler schreibst du mehrere `except`-Blöcke. Sollen mehrere Fehler gleich behandelt werden, zählst du sie in einem `except` auf. Seit Python 3.14 brauchst du dafür keine Klammern mehr, solange kein `as` folgt:

In [ ]:
def to_number(value):
    try:
        return int(value)
    except ValueError, TypeError:  # "drei" ist ein ValueError, None ein TypeError
        return None


print(to_number("42"), to_number("drei"), to_number(None))

## Fehlermeldungen helfen mit

Aktuelle Python-Versionen schlagen bei Tippfehlern vor, was du gemeint haben könntest. Wer aus Java oder JavaScript kommt, schreibt gern `push` statt `append`. Lies deshalb immer die letzte Zeile des Tracebacks bis zum Ende:

In [ ]:
numbers = [1, 2]
numbers.push(3)

## `else` und `finally`

- `else` läuft nur, wenn **kein** Fehler aufgetreten ist.
- `finally` läuft **immer**, z. B. zum Aufräumen.

In [ ]:
def parse_number(text):
    try:
        number = float(text)
    except ValueError:
        print(f"'{text}' ist ungültig")
        return None
    else:
        print(f"'{text}' gelesen")
        return number
    finally:
        print("Prüfung beendet")


parse_number("3.5")
parse_number("abc")

## Eingaben robust machen

Ein typisches Muster: so lange fragen, bis die Eingabe passt.

In [ ]:
while True:
    try:
        age = int(input("Alter: "))
        break
    except ValueError:
        print("Bitte eine ganze Zahl eingeben.")
print(f"Alter: {age}")

**Übung 2 (Sensor-Export bereinigen, mittel):** Ein Sensor-Export liefert `readings` als Text, darunter `"n/a"`, leere Werte, Dezimalkommas und einen unmöglichen Wert. Wandle alle gültigen Werte in `float` um, zähle die ungültigen und gib den Mittelwert der gültigen aus. Ein ungültiger Wert darf den Rest nicht abbrechen.

In [ ]:
readings = ["23.5", "n/a", "21,7", "", "22.0", "-273.5"]

In [ ]:
valid, invalid = [], 0
for raw in readings:
    try:
        value = float(raw.replace(",", "."))
    except ValueError:
        invalid += 1
        continue
    if -40 <= value <= 60:  # plausibel für einen Raumsensor
        valid.append(value)
    else:
        invalid += 1
print(f"{len(valid)} gültig, {invalid} ungültig, Schnitt {sum(valid) / len(valid):.1f} °C")

## Fehler selbst auslösen mit `raise`

Bekommt deine Funktion ungültige Werte, meldet sie das mit `raise`. Der Aufrufer entscheidet, wie er damit umgeht.

In [ ]:
def withdraw(balance, amount):
    if amount <= 0:
        raise ValueError(f"Betrag muss positiv sein, war {amount}")
    if amount > balance:
        raise ValueError("Kontostand reicht nicht aus")
    return balance - amount


print(withdraw(100, 30))
print(withdraw(100, 300))

## Eigene Fehlerklassen

Für eigene Fehlerarten erbst du von einer passenden Fehlerklasse. Mehr ist oft nicht nötig:

In [ ]:
class InsufficientFundsError(ValueError):
    """Wird ausgelöst, wenn das Konto nicht gedeckt ist."""


def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(f"{amount} € angefordert, nur {balance} € verfügbar")
    return balance - amount


try:
    withdraw(100, 300)
except InsufficientFundsError as error:
    print(f"Abgelehnt: {error}")

> Wie Klassen und Vererbung im Detail funktionieren, lernst du in Notebook 12.

## Ressourcen sicher schließen mit `with`

Dateien, Netzwerkverbindungen usw. müssen geschlossen werden, auch wenn ein Fehler auftritt. `with` erledigt das automatisch. Mehr dazu in Notebook 10.

In [ ]:
try:
    with open("../daten/numbers.txt", encoding="utf-8") as file:
        count = int(file.readline())
        value = float(file.readline())
    print(count, value)
except FileNotFoundError:
    print("Datei nicht gefunden")
except ValueError:
    print("Keine gültige Zahl in Zeile 1 oder 2")

## Übungen

**Übung 3 (Bestellzeile prüfen, leicht):** Die Zeilen unten prüfen eine Bestellzeile: gültig heißt Ausgabe des Betrags, ungültig heißt verständliche Meldung. Bringe sie in die richtige Reihenfolge und rücke sie richtig ein. Eine Zeile wird nicht gebraucht. Teste danach mit `"Kaffee;3;4.20"` und `"Tee;zwei;3.10"`.

```text
else:
print(f"Ungültige Zeile: {line}")
line = "Tee;zwei;3.10"
amount = int(quantity) * float(price)
except ValueError:
product, quantity, price = line.split(";")
print(f"{product}: {amount:.2f} €")
try:
return amount
```

In [ ]:
for line in ["Kaffee;3;4.20", "Tee;zwei;3.10"]:  # in der Aufgabe je eine Zeile
    product, quantity, price = line.split(";")
    try:
        amount = int(quantity) * float(price)
    except ValueError:
        print(f"Ungültige Zeile: {line}")
    else:
        print(f"{product}: {amount:.2f} €")
# Störzeile "return amount": return gibt es nur in Funktionen

**Übung 4 (IBAN prüfen, schwer):** Schreibe `validate_iban(text)` für deutsche IBANs (22 Zeichen, beginnt mit `DE`, Prüfsumme modulo 97 ergibt 1) und eine eigene Fehlerklasse `InvalidIbanError(ValueError)` mit einer Meldung je Fehlerart. Teste mit `"DE89 3704 0044 0532 0130 00"` und derselben IBAN mit Zahlendreher. Tipp: `int("D", 36)` ergibt 13.

In [ ]:
class InvalidIbanError(ValueError):
    pass


def validate_iban(text: str) -> str:
    iban = text.replace(" ", "").upper()
    if len(iban) != 22 or not iban.startswith("DE"):
        raise InvalidIbanError("Deutsche IBAN: 22 Zeichen, beginnt mit DE")
    rearranged = iban[4:] + iban[:4]
    if int("".join(str(int(char, 36)) for char in rearranged)) % 97 != 1:
        raise InvalidIbanError("Prüfsumme stimmt nicht, Tippfehler?")
    return iban


for candidate in ["DE89 3704 0044 0532 0130 00", "DE89 3704 0044 0532 0310 00"]:
    try:
        print("gültig:", validate_iban(candidate))
    except ValueError as error:
        print("ungültig:", error)

## Zusammenfassung

- Tracebacks von unten nach oben lesen.
- `try` / `except Fehlertyp` fängt konkrete Fehler ab, `as error` liefert Details.
- `else` läuft ohne Fehler, `finally` immer.
- `raise ValueError("…")` meldet ungültige Werte.
- Eigene Fehler: `class MyError(ValueError): ...`
- `with` schließt Ressourcen automatisch.